In [ ]:
import sys
import os
from pathlib import Path


def _add_if_repo_root(path: Path) -> bool:
    path = path.resolve()
    if (path / "src" / "ml" / "retrain_gate.py").exists() and (path / "src" / "ml" / "__init__.py").exists():
        if str(path) not in sys.path:
            sys.path.insert(0, str(path))
        return True
    return False


# 1) Current working directory and parents.
# On newer Databricks runtimes, the notebook/script directory is commonly the CWD.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if _add_if_repo_root(candidate):
        print(f"OK: added repo root from cwd search: {candidate}")
        break
else:
    # 2) Resolve from notebook workspace path.
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    notebook_path = ctx.notebookPath().get()

    # notebook_path is usually like:
    # /Users/<user>/.bundle/healthcare-claim-ops/dev/files/src/notebooks/check_new_data
    workspace_notebook_path = Path("/Workspace") / notebook_path.lstrip("/")

    found = False
    for candidate in [workspace_notebook_path, *workspace_notebook_path.parents]:
        if _add_if_repo_root(candidate):
            print(
                f"OK: added repo root from notebook path search: {candidate}")
            found = True
            break

    if not found:
        print("DEBUG cwd:", Path.cwd())
        print("DEBUG notebook_path:", notebook_path)
        print("DEBUG workspace_notebook_path:", workspace_notebook_path)
        print("DEBUG first sys.path entries:", sys.path[:10])
        raise RuntimeError(
            "Could not locate repo root containing src/ml/retrain_gate.py")


print("OK: imported src.ml.retrain_gate and FEATURE_COLUMNS")

In [ ]:
from src.ml import FEATURE_COLUMNS
from src.ml.retrain_gate import _current_gold_object_metadata, decide_retrain

dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")
dbutils.widgets.text("ml_schema", "ml", "ML schema")
dbutils.widgets.text("registered_model_name",
                     "healthcare.ml.claim_denial_model", "Registered model name")
dbutils.widgets.text("champion_alias", "champion", "Champion alias")

catalog = dbutils.widgets.get("catalog").strip()
gold_schema = dbutils.widgets.get("gold_schema").strip()
ml_schema = dbutils.widgets.get("ml_schema").strip()
registered_model_name = dbutils.widgets.get("registered_model_name").strip()
champion_alias = dbutils.widgets.get("champion_alias").strip()
gold_table = f"{catalog}.{gold_schema}.claim_features"

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
)

# 1. Compute retrain decision FIRST.
decision = decide_retrain(
    spark,
    gold_table=gold_table,
    feature_columns=list(FEATURE_COLUMNS),
    registered_model_name=registered_model_name,
    champion_alias=champion_alias,
)

print(decision.summary_line())

# 2. Resolve Gold object metadata.
gold_obj_type, gold_last_altered = _current_gold_object_metadata(spark, gold_table)

# 3. Write audit row for every decision status.
decision_schema = StructType(
    [
        StructField("decision_status", StringType(), False),
        StructField("should_retrain", StringType(), True),
        StructField("reason", StringType(), False),
        StructField("error_detail", StringType(), True),
        StructField("current_row_count", LongType(), False),
        StructField("current_gold_version", LongType(), True),
        StructField("current_gold_object_type", StringType(), True),
        StructField("current_gold_last_altered", StringType(), True),
        StructField("current_fingerprint", StringType(), False),
        StructField("champion_run_id", StringType(), True),
        StructField("previous_training_row_count", LongType(), True),
        StructField("row_count_delta", LongType(), True),
        StructField("row_count_delta_pct", DoubleType(), True),
    ]
)

def _maybe_int(value):
    return int(value) if value is not None else None

def _maybe_float(value):
    return float(value) if value is not None else None

def _maybe_str(value):
    return str(value) if value is not None else None

decision_df = spark.createDataFrame(
    [
        (
            str(decision.decision_status),
            str(decision.should_retrain).lower() if decision.should_retrain is not None else None,
            str(decision.reason),
            _maybe_str(decision.error_detail),
            int(decision.current_row_count),
            _maybe_int(decision.current_gold_version),
            _maybe_str(gold_obj_type),
            _maybe_str(gold_last_altered),
            str(decision.current_fingerprint),
            _maybe_str(decision.champion_run_id),
            _maybe_int(decision.previous_training_row_count),
            _maybe_int(decision.row_count_delta),
            _maybe_float(decision.row_count_delta_pct),
        )
    ],
    schema=decision_schema,
)

decision_df.createOrReplaceTempView("_current_retrain_decision")

spark.sql(f"""
INSERT INTO {catalog}.{ml_schema}.retrain_decisions
  (decided_at, decision_status, should_retrain, reason, error_detail,
   current_row_count, current_gold_version, current_gold_object_type,
   current_gold_last_altered, current_fingerprint, champion_run_id,
   previous_training_row_count, row_count_delta, row_count_delta_pct)
SELECT
  current_timestamp() AS decided_at,
  decision_status,
  should_retrain,
  reason,
  error_detail,
  current_row_count,
  current_gold_version,
  current_gold_object_type,
  current_gold_last_altered,
  current_fingerprint,
  champion_run_id,
  previous_training_row_count,
  row_count_delta,
  row_count_delta_pct
FROM _current_retrain_decision
""")

print(f"OK: audit row written for decision_status={decision.decision_status}")

In [ ]:
# 4. Only set task values for retrain and skip states.
#    On error, exit nonzero so the job stops and alerts.
if decision.decision_status == "error":
    print(f"ERROR: retrain gate failed: {decision.reason}")
    if decision.error_detail:
        print(f"  detail: {decision.error_detail}")
    sys.exit(1)

dbutils.jobs.taskValues.set(
    key="should_retrain",
    value=str(decision.should_retrain).lower(),
)
dbutils.jobs.taskValues.set(
    key="reason",
    value=decision.reason,
)
dbutils.jobs.taskValues.set(
    key="current_training_row_count",
    value=str(decision.current_row_count),
)
dbutils.jobs.taskValues.set(
    key="current_gold_version",
    value=str(decision.current_gold_version),
)
dbutils.jobs.taskValues.set(
    key="current_data_fingerprint",
    value=decision.current_fingerprint,
)

print(f"OK: task values set for decision_status={decision.decision_status}")